# 01 - Data Collection

This notebook collects data from two sources:
1. **elap733/NBA-Injuries-Analysis** - Pre-scraped injury transaction data (2010-2019)
2. **nba_api** - Official NBA stats API for player statistics, bio data, tracking data, and schedules

All raw data is saved to `data/raw/` subdirectories.

---

## Final Feature Set (Selected After EDA)

After exploratory analysis, these features showed strongest correlation with injury (games_missed):

| Feature | r | Source | Description |
|---------|---|--------|-------------|
| **age_x_min** | 0.235 | Derived | Age × Minutes interaction (older + heavy workload = risk) |
| **min** | 0.220 | nba_api | Minutes per game |
| **b2b_games** | 0.216 | Derived | Team's back-to-back games from schedule |
| **games_missed_last_season** | 0.213 | elap733 | Prior season injury history |
| **b2b_percentage** | 0.206 | Derived | % of games that are back-to-backs |
| **age** | 0.094 | nba_api | Player age |
| **position_group** | — | Derived | Guard/Forward/Center (from height) |

**Note:** Hustle stats (contested shots, deflections) showed r=0.19-0.21 but only available 2016+. Dropped to maintain full 2010-2019 dataset coverage.

In [5]:
import sys
sys.path.append('..')

import os
import time
import pandas as pd
import numpy as np
from pathlib import Path

from src.config import (
    FIRST_SEASON, LAST_SEASON,
    TRACKING_DATA_START,
    RAW_ELAP_DIR, RAW_NBA_API_DIR
)

# Dataframe display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

In [6]:
# Ensuring output directories exist
Path(f'../{RAW_ELAP_DIR}').mkdir(parents=True, exist_ok=True)
Path(f'../{RAW_NBA_API_DIR}').mkdir(parents=True, exist_ok=True)

print(f"Season range: {FIRST_SEASON} to {LAST_SEASON}")
print(f"Tracking data available from: {TRACKING_DATA_START}")

Season range: 2010 to 2019
Tracking data available from: 2013


In [7]:
def safe_api_call(endpoint_class, max_retries=3, **kwargs):
    """
    Call nba_api endpoint with rate limiting and retry logic.
    
    Parameters
    ----------
    endpoint_class : class
        nba_api endpoint class to instantiate
    max_retries : int
        Maximum number of retry attempts
    **kwargs : dict
        Arguments to pass to the endpoint
    
    Returns
    -------
    pd.DataFrame
        First DataFrame from the endpoint response
    """
    time.sleep(1.5)  # Rate limiting
    
    for attempt in range(max_retries):
        try:
            result = endpoint_class(**kwargs)
            dfs = result.get_data_frames()
            if dfs:
                return dfs[0]
            return pd.DataFrame()
        except Exception as e:
            print(f"  Attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                wait_time = 5 * (attempt + 1)  
                print(f"  Waiting {wait_time}s before retry...")
                time.sleep(wait_time)
            else:
                print(f"  All retries failed, returning empty DataFrame")
                return pd.DataFrame()

---
## Section 1: Load elap733 Injury Data

The [elap733/NBA-Injuries-Analysis](https://github.com/elap733/NBA-Injuries-Analysis) repository contains pre-scraped injury transaction data from prosportstransactions.com (2010-2019).

**Files downloaded to `data/raw/elap733/`:**
- missed_games_2010_2019.csv - **Games missed per injury event (our target variable source)**
- injury_transactions_2010_2019.csv - Inactive Reserve List (IRL) transaction records
- player_stats_1994_2019.csv - Historical player stats
- team_schedules_2010_2020.csv - Team game schedules

In [8]:
# Load missed games data - OUR TARGET VARIABLE SOURCE
df_missed_games = pd.read_csv(f'../{RAW_ELAP_DIR}/missed_games_2010_2019.csv')

print("=" * 60)
print("MISSED GAMES DATA (Target Variable Source)")
print("=" * 60)
print(f"Shape: {df_missed_games.shape}")
print(f"\nColumns: {list(df_missed_games.columns)}")
print(f"\nFirst 5 rows:")
df_missed_games.head()

MISSED GAMES DATA (Target Variable Source)
Shape: (11531, 6)

Columns: ['Unnamed: 0', 'Date', 'Team', 'Acquired', 'Relinquished', 'Notes']

First 5 rows:


,Unnamed: 0,Date,Team,Acquired,Relinquished,Notes
0,0,2010-08-03,Clippers,NaN,Craig Smith,arthroscopic surgery on right knee (out indefi...
1,1,2010-08-13,Mavericks,NaN,Rodrigue Beaubois,surgery on left foot to repair broken fifth me...
2,2,2010-08-14,Warriors,NaN,Ekpe Udoh,surgery on left wrist (out indefinitely)
3,3,2010-09-21,Raptors,NaN,Ed Davis (a),arthroscopic surgery on right kene to repair t...
4,4,2010-09-21,Thunder,NaN,Nenad Krstic,surgery on right hand to repair broken finger ...


In [5]:
# Explore missed games data structure
print("\nData types:")
print(df_missed_games.dtypes)
print(f"\nMissing values:")
print(df_missed_games.isnull().sum())
print(f"\nBasic stats for numeric columns:")
df_missed_games.describe()


Data types:
Unnamed: 0       int64
Date            object
Team            object
Acquired        object
Relinquished    object
Notes           object
dtype: object

Missing values:
Unnamed: 0         0
Date               0
Team               2
Acquired        9328
Relinquished    2203
Notes              0
dtype: int64

Basic stats for numeric columns:


,Unnamed: 0
count,11531.000000
mean,5765.000000
std,3328.857311
min,0.000000
25%,2882.500000
50%,5765.000000
75%,8647.500000
max,11530.000000


In [9]:
# Load injury transactions data from elap733 repository
df_injury_transactions = pd.read_csv(f'../{RAW_ELAP_DIR}/injury_transactions_2010_2019.csv')

print("=" * 60)
print("INJURY TRANSACTIONS DATA")
print("=" * 60)
print(f"Shape: {df_injury_transactions.shape}")
print(f"\nColumns: {list(df_injury_transactions.columns)}")
print(f"\nFirst 5 rows:")
df_injury_transactions.head()

INJURY TRANSACTIONS DATA
Shape: (14135, 6)

Columns: ['Unnamed: 0', 'Date', 'Team', 'Acquired', 'Relinquished', 'Notes']

First 5 rows:


,Unnamed: 0,Date,Team,Acquired,Relinquished,Notes
0,0,2010-10-26,Blazers,NaN,Elliot Williams,placed on IL
1,1,2010-10-26,Blazers,NaN,Greg Oden,placed on IL with left knee injury (out for se...
2,2,2010-10-26,Blazers,NaN,Joel Przybilla,placed on IL placed on IL recovering from surg...
3,3,2010-10-26,Celtics,NaN,Avery Bradley,placed on IL recovering from surgery on left a...
4,4,2010-10-26,Celtics,NaN,Kendrick Perkins,placed on IL recovering from surgery on right ...


In [7]:
# Load elap733 player stats (for reference/comparison)
df_elap_stats = pd.read_csv(f'../{RAW_ELAP_DIR}/player_stats_1994_2019.csv')

print("=" * 60)
print("ELAP733 PLAYER STATS DATA")
print("=" * 60)
print(f"Shape: {df_elap_stats.shape}")
print(f"\nColumns: {list(df_elap_stats.columns)}")
print(f"\nFirst 5 rows:")
df_elap_stats.head()

ELAP733 PLAYER STATS DATA
Shape: (19786, 32)

Columns: ['Unnamed: 0', 'Year', 'Season', 'Player', 'Pos', 'Age', 'Tm', 'G', 'GS', 'MP', 'FG', 'FGA', 'FG%', '3P', '3PA', '3P%', '2P', '2PA', '2P%', 'eFG%', 'FT', 'FTA', 'FT%', 'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS']

First 5 rows:


,Unnamed: 0,Year,Season,Player,Pos,Age,Tm,G,GS,MP,FG,FGA,FG%,3P,3PA,3P%,2P,2PA,2P%,eFG%,FT,FTA,FT%,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS
0,0,1993,regular,Alaa Abdelnaby,PF,25,BOS,13,0,12.2,1.8,4.2,0.436,0.0,0.0,NaN,1.8,4.2,0.436,0.436,1.2,1.9,0.640,0.9,2.6,3.5,0.2,0.2,0.2,1.3,1.5,4.9
1,1,1993,regular,Mahmoud Abdul-Rauf,PG,24,DEN,80,78,32.7,7.4,16.0,0.460,0.5,1.7,0.316,6.8,14.3,0.476,0.476,2.7,2.9,0.956,0.3,1.8,2.1,4.5,1.0,0.1,1.9,1.9,18.0
2,2,1993,regular,Michael Adams,PG,31,WSB,70,67,33.4,4.1,10.0,0.408,0.8,2.7,0.288,3.3,7.2,0.454,0.448,3.2,3.9,0.830,0.5,2.1,2.6,6.9,1.4,0.1,2.4,2.0,12.1
3,3,1993,regular,Mark Aguirre,SF,34,LAC,39,0,22.0,4.2,8.9,0.468,0.9,2.4,0.398,3.2,6.5,0.494,0.522,1.3,1.8,0.694,0.7,2.3,3.0,2.7,0.5,0.2,1.8,2.5,10.6
4,4,1993,regular,Danny Ainge,SG,34,PHO,68,1,22.9,3.3,7.9,0.417,1.2,3.6,0.328,2.1,4.3,0.491,0.492,1.1,1.4,0.830,0.4,1.5,1.9,2.6,0.8,0.1,1.2,2.1,8.9


In [8]:
# Load elap733 team schedules
df_elap_schedules = pd.read_csv(f'../{RAW_ELAP_DIR}/team_schedules_2010_2020.csv')

print("=" * 60)
print("ELAP733 TEAM SCHEDULES DATA")
print("=" * 60)
print(f"Shape: {df_elap_schedules.shape}")
print(f"\nColumns: {list(df_elap_schedules.columns)}")
print(f"\nFirst 5 rows:")
df_elap_schedules.head()

ELAP733 TEAM SCHEDULES DATA
Shape: (28240, 9)

Columns: ['Unnamed: 0', 'Team', 'Year', 'Season', 'Game_num', 'Date', 'Away_flag', 'Opponent', 'OT_flag']

First 5 rows:


,Unnamed: 0,Team,Year,Season,Game_num,Date,Away_flag,Opponent,OT_flag
0,0,Atlanta Hawks,2009,regular,1,2009-10-28,0,Indiana Pacers,NaN
1,1,Atlanta Hawks,2009,regular,2,2009-10-30,0,Washington Wizards,NaN
2,2,Atlanta Hawks,2009,regular,3,2009-11-01,1,Los Angeles Lakers,NaN
3,3,Atlanta Hawks,2009,regular,4,2009-11-03,1,Portland Trail Blazers,NaN
4,4,Atlanta Hawks,2009,regular,5,2009-11-04,1,Sacramento Kings,NaN


### Section 1 Summary

**Key finding:** The `missed_games_2010_2019.csv` file contains our target variable: games missed per injury event. This data was scraped from prosportstransactions.com and includes player name, team, date, injury notes, and games missed.

In the following data cleaning notebook, we'll:
- Aggregate games missed by player and season to create `games_missed_next_season` target
- Parse injury notes to extract injury type/body part
- Standardize player names for matching with nba_api data

---
## Section 2: Pull Player Season Stats via nba_api

Use [nba_api](https://github.com/swar/nba_api) to pull player per-game statistics for each season from 2010-2019.

**Endpoint:** `LeagueDashPlayerStats`

**Key fields:** Player ID, name, team, games played, minutes, points, rebounds, assists, usage rate, etc.

**Output:** `data/raw/nba_api/player_stats_{season}.csv` for each season

In [9]:
from nba_api.stats.endpoints import leaguedashplayerstats

print("Pulling player season stats from nba_api...")
print(f"Seasons: {FIRST_SEASON}-{str(FIRST_SEASON+1)[-2:]} through {LAST_SEASON}-{str(LAST_SEASON+1)[-2:]}")
print("="*60)

for season in range(FIRST_SEASON, LAST_SEASON + 1):
    season_str = f"{season}-{str(season+1)[-2:]}"  # e.g., "2010-11"
    filepath = f"../{RAW_NBA_API_DIR}/player_stats_{season}.csv"
    
    # Check if already exists
    if os.path.exists(filepath):
        print(f"[SKIP] {season_str}: Already exists at {filepath}")
        continue
    
    print(f"[FETCH] {season_str}...", end=" ")
    
    df = safe_api_call(
        leaguedashplayerstats.LeagueDashPlayerStats,
        season=season_str,
        per_mode_detailed='PerGame'
    )
    
    if not df.empty:
        df['SEASON'] = season_str
        df['SEASON_START_YEAR'] = season
        df.to_csv(filepath, index=False)
        print(f"Saved {len(df)} players")
    else:
        print("No data returned")

print("\nDone with player season stats!")

Pulling player season stats from nba_api...
Seasons: 2010-11 through 2019-20
[FETCH] 2010-11... Saved 452 players
[FETCH] 2011-12... Saved 478 players
[FETCH] 2012-13... Saved 469 players
[FETCH] 2013-14... Saved 482 players
[FETCH] 2014-15... Saved 492 players
[FETCH] 2015-16... Saved 476 players
[FETCH] 2016-17... Saved 486 players
[FETCH] 2017-18... Saved 540 players
[FETCH] 2018-19... Saved 530 players
[FETCH] 2019-20... Saved 529 players

Done with player season stats!


In [10]:
# Verify saved player stats files
print("Saved player stats files:")
for season in range(FIRST_SEASON, LAST_SEASON + 1):
    filepath = f"../{RAW_NBA_API_DIR}/player_stats_{season}.csv"
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        print(f"  player_stats_{season}.csv: {df.shape[0]} players, {df.shape[1]} columns")
    else:
        print(f"  player_stats_{season}.csv: NOT FOUND")

Saved player stats files:
  player_stats_2010.csv: 452 players, 69 columns
  player_stats_2011.csv: 478 players, 69 columns
  player_stats_2012.csv: 469 players, 69 columns
  player_stats_2013.csv: 482 players, 69 columns
  player_stats_2014.csv: 492 players, 69 columns
  player_stats_2015.csv: 476 players, 69 columns
  player_stats_2016.csv: 486 players, 69 columns
  player_stats_2017.csv: 540 players, 69 columns
  player_stats_2018.csv: 530 players, 69 columns
  player_stats_2019.csv: 529 players, 69 columns


In [11]:
# Preview sample player stats file
sample_file = f"../{RAW_NBA_API_DIR}/player_stats_{FIRST_SEASON}.csv"
if os.path.exists(sample_file):
    df_sample = pd.read_csv(sample_file)
    print(f"Sample: player_stats_{FIRST_SEASON}.csv")
    print(f"Columns: {list(df_sample.columns)}")
    print(f"\nFirst 3 rows:")
    display(df_sample.head(3))

Sample: player_stats_2010.csv
Columns: ['PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'AGE', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS', 'NBA_FANTASY_PTS', 'DD2', 'TD3', 'WNBA_FANTASY_PTS', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK', 'FGM_RANK', 'FGA_RANK', 'FG_PCT_RANK', 'FG3M_RANK', 'FG3A_RANK', 'FG3_PCT_RANK', 'FTM_RANK', 'FTA_RANK', 'FT_PCT_RANK', 'OREB_RANK', 'DREB_RANK', 'REB_RANK', 'AST_RANK', 'TOV_RANK', 'STL_RANK', 'BLK_RANK', 'BLKA_RANK', 'PF_RANK', 'PFD_RANK', 'PTS_RANK', 'PLUS_MINUS_RANK', 'NBA_FANTASY_PTS_RANK', 'DD2_RANK', 'TD3_RANK', 'WNBA_FANTASY_PTS_RANK', 'TEAM_COUNT', 'SEASON', 'SEASON_START_YEAR']

First 3 rows:


,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,AGE,GP,W,L,W_PCT,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,...,FG3M_RANK,FG3A_RANK,FG3_PCT_RANK,FTM_RANK,FTA_RANK,FT_PCT_RANK,OREB_RANK,DREB_RANK,REB_RANK,AST_RANK,TOV_RANK,STL_RANK,BLK_RANK,BLKA_RANK,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,NBA_FANTASY_PTS_RANK,DD2_RANK,TD3_RANK,WNBA_FANTASY_PTS_RANK,TEAM_COUNT,SEASON,SEASON_START_YEAR
0,201985,AJ Price,AJ,1610612754,IND,24.0,50,22,28,0.440,15.9,2.3,6.4,0.356,0.8,3.0,0.275,1.1,1.6,0.667,0.3,1.1,1.4,2.2,1.1,...,137,97,241,234,212,316,348,369,375,121,203,208,421,308,351,198,240,161,274,224,27,264,1,2010-11,2010
1,201166,Aaron Brooks,Aaron,1610612756,PHX,26.0,59,26,33,0.441,21.8,3.7,9.9,0.375,1.2,4.0,0.297,2.1,2.4,0.886,0.3,1.0,1.3,3.9,1.7,...,86,55,227,104,138,30,344,388,385,44,99,209,392,62,205,131,132,377,179,154,27,164,2,2010-11,2010
2,201189,Aaron Gray,Aaron,1610612740,NOH,26.0,41,21,20,0.512,12.9,1.4,2.4,0.566,0.0,0.0,0.000,0.4,0.8,0.500,1.4,2.7,4.2,0.4,0.8,...,309,383,309,367,333,398,98,169,133,376,290,368,223,270,121,348,362,145,326,154,27,332,1,2010-11,2010


---
## Section 3: Pull Player Bio Data via nba_api

Collect player biographical/demographic information for all players active during our date range.

**Endpoint:** `LeagueDashPlayerBioStats`

**Key fields:** Player ID, name, age, height, weight, position, draft year, experience

**Output:** `data/raw/nba_api/player_bio_{season}.csv` for each season

In [12]:
from nba_api.stats.endpoints import leaguedashplayerbiostats

print("Pulling player bio stats from nba_api...")
print("="*60)

for season in range(FIRST_SEASON, LAST_SEASON + 1):
    season_str = f"{season}-{str(season+1)[-2:]}"
    filepath = f"../{RAW_NBA_API_DIR}/player_bio_{season}.csv"
    
    if os.path.exists(filepath):
        print(f"[SKIP] {season_str}: Already exists")
        continue
    
    print(f"[FETCH] {season_str}...", end=" ")
    
    df = safe_api_call(
        leaguedashplayerbiostats.LeagueDashPlayerBioStats,
        season=season_str,
        per_mode_simple='PerGame'
    )
    
    if not df.empty:
        df['SEASON'] = season_str
        df['SEASON_START_YEAR'] = season
        df.to_csv(filepath, index=False)
        print(f"Saved {len(df)} players")
    else:
        print("No data returned")

print("\nDone with player bio stats!")

Pulling player bio stats from nba_api...
[FETCH] 2010-11... Saved 452 players
[FETCH] 2011-12... Saved 478 players
[FETCH] 2012-13... Saved 469 players
[FETCH] 2013-14... Saved 482 players
[FETCH] 2014-15... Saved 492 players
[FETCH] 2015-16... Saved 476 players
[FETCH] 2016-17... Saved 486 players
[FETCH] 2017-18... Saved 540 players
[FETCH] 2018-19... Saved 530 players
[FETCH] 2019-20... Saved 529 players

Done with player bio stats!


In [13]:
# Verify saved player bio files
print("Saved player bio files:")
for season in range(FIRST_SEASON, LAST_SEASON + 1):
    filepath = f"../{RAW_NBA_API_DIR}/player_bio_{season}.csv"
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        print(f"  player_bio_{season}.csv: {df.shape[0]} players, {df.shape[1]} columns")
    else:
        print(f"  player_bio_{season}.csv: NOT FOUND")

Saved player bio files:
  player_bio_2010.csv: 452 players, 25 columns
  player_bio_2011.csv: 478 players, 25 columns
  player_bio_2012.csv: 469 players, 25 columns
  player_bio_2013.csv: 482 players, 25 columns
  player_bio_2014.csv: 492 players, 25 columns
  player_bio_2015.csv: 476 players, 25 columns
  player_bio_2016.csv: 486 players, 25 columns
  player_bio_2017.csv: 540 players, 25 columns
  player_bio_2018.csv: 530 players, 25 columns
  player_bio_2019.csv: 529 players, 25 columns


In [14]:
# Preview sample player bio file
sample_file = f"../{RAW_NBA_API_DIR}/player_bio_{FIRST_SEASON}.csv"
if os.path.exists(sample_file):
    df_sample = pd.read_csv(sample_file)
    print(f"Sample: player_bio_{FIRST_SEASON}.csv")
    print(f"Columns: {list(df_sample.columns)}")
    print(f"\nFirst 3 rows:")
    display(df_sample.head(3))

Sample: player_bio_2010.csv
Columns: ['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'AGE', 'PLAYER_HEIGHT', 'PLAYER_HEIGHT_INCHES', 'PLAYER_WEIGHT', 'COLLEGE', 'COUNTRY', 'DRAFT_YEAR', 'DRAFT_ROUND', 'DRAFT_NUMBER', 'GP', 'PTS', 'REB', 'AST', 'NET_RATING', 'OREB_PCT', 'DREB_PCT', 'USG_PCT', 'TS_PCT', 'AST_PCT', 'SEASON', 'SEASON_START_YEAR']

First 3 rows:


,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_ABBREVIATION,AGE,PLAYER_HEIGHT,PLAYER_HEIGHT_INCHES,PLAYER_WEIGHT,COLLEGE,COUNTRY,DRAFT_YEAR,DRAFT_ROUND,DRAFT_NUMBER,GP,PTS,REB,AST,NET_RATING,OREB_PCT,DREB_PCT,USG_PCT,TS_PCT,AST_PCT,SEASON,SEASON_START_YEAR
0,201985,AJ Price,1610612754,IND,24.0,6-2,74,181,Connecticut,USA,2009,2,52,50,6.5,1.4,2.2,2.2,0.019,0.073,0.223,0.454,0.233,2010-11,2010
1,201166,Aaron Brooks,1610612756,PHX,26.0,6-0,72,161,Oregon,USA,2007,1,26,59,10.7,1.3,3.9,-6.6,0.015,0.046,0.251,0.489,0.289,2010-11,2010
2,201189,Aaron Gray,1610612740,NOH,26.0,7-0,84,270,Pittsburgh,USA,2007,2,49,41,3.1,4.2,0.4,3.9,0.121,0.236,0.130,0.566,0.044,2010-11,2010


---
## Section 4: Pull Tracking Data via nba_api (2013-14+)

NBA.com tracking data (player speed, distance, drives, touches) is only available from the 2013-14 season onward.

**Endpoint:** `LeagueDashPtStats`

**Key fields:** Speed, distance traveled, touches, drives, passes, contested shots

**Output:** `data/raw/nba_api/tracking_stats_{season}.csv` for seasons 2013-2019

In [15]:
from nba_api.stats.endpoints import leaguedashptstats

print("Pulling player tracking stats from nba_api...")
print(f"Note: Tracking data only available from {TRACKING_DATA_START}-{str(TRACKING_DATA_START+1)[-2:]} onward")
print("="*60)

# Tracking stats have different player tracking types
# We'll pull 'SpeedDistance' which gives speed and distance metrics

for season in range(TRACKING_DATA_START, LAST_SEASON + 1):
    season_str = f"{season}-{str(season+1)[-2:]}"
    filepath = f"../{RAW_NBA_API_DIR}/tracking_stats_{season}.csv"
    
    if os.path.exists(filepath):
        print(f"[SKIP] {season_str}: Already exists")
        continue
    
    print(f"[FETCH] {season_str}...", end=" ")
    
    df = safe_api_call(
        leaguedashptstats.LeagueDashPtStats,
        season=season_str,
        per_mode_simple='PerGame',
        player_or_team='Player',
        pt_measure_type='SpeedDistance'
    )
    
    if not df.empty:
        df['SEASON'] = season_str
        df['SEASON_START_YEAR'] = season
        df.to_csv(filepath, index=False)
        print(f"Saved {len(df)} players")
    else:
        print("No data returned")

print("\nDone with tracking stats!")

Pulling player tracking stats from nba_api...
Note: Tracking data only available from 2013-14 onward
[FETCH] 2013-14... Saved 482 players
[FETCH] 2014-15... Saved 492 players
[FETCH] 2015-16... Saved 476 players
[FETCH] 2016-17... Saved 486 players
[FETCH] 2017-18... Saved 540 players
[FETCH] 2018-19... Saved 530 players
[FETCH] 2019-20... Saved 529 players

Done with tracking stats!


In [16]:
# Verify saved tracking files
print("Saved tracking stats files:")
for season in range(TRACKING_DATA_START, LAST_SEASON + 1):
    filepath = f"../{RAW_NBA_API_DIR}/tracking_stats_{season}.csv"
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        print(f"  tracking_stats_{season}.csv: {df.shape[0]} players, {df.shape[1]} columns")
    else:
        print(f"  tracking_stats_{season}.csv: NOT FOUND")

Saved tracking stats files:
  tracking_stats_2013.csv: 482 players, 18 columns
  tracking_stats_2014.csv: 492 players, 18 columns
  tracking_stats_2015.csv: 476 players, 18 columns
  tracking_stats_2016.csv: 486 players, 18 columns
  tracking_stats_2017.csv: 540 players, 18 columns
  tracking_stats_2018.csv: 530 players, 18 columns
  tracking_stats_2019.csv: 529 players, 18 columns


In [17]:
# Preview sample tracking file
sample_file = f"../{RAW_NBA_API_DIR}/tracking_stats_{TRACKING_DATA_START}.csv"
if os.path.exists(sample_file):
    df_sample = pd.read_csv(sample_file)
    print(f"Sample: tracking_stats_{TRACKING_DATA_START}.csv")
    print(f"Columns: {list(df_sample.columns)}")
    print(f"\nFirst 3 rows:")
    display(df_sample.head(3))

Sample: tracking_stats_2013.csv
Columns: ['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'GP', 'W', 'L', 'MIN', 'MIN1', 'DIST_FEET', 'DIST_MILES', 'DIST_MILES_OFF', 'DIST_MILES_DEF', 'AVG_SPEED', 'AVG_SPEED_OFF', 'AVG_SPEED_DEF', 'SEASON', 'SEASON_START_YEAR']

First 3 rows:


,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_ABBREVIATION,GP,W,L,MIN,MIN1,DIST_FEET,DIST_MILES,DIST_MILES_OFF,DIST_MILES_DEF,AVG_SPEED,AVG_SPEED_OFF,AVG_SPEED_DEF,SEASON,SEASON_START_YEAR
0,201985,AJ Price,1610612750,MIN,28,15,13,3.5,3.5,1350.5,0.3,0.1,0.1,4.37,4.62,3.96,2013-14,2013
1,201166,Aaron Brooks,1610612743,DEN,72,42,30,21.6,21.6,8027.7,1.5,0.8,0.8,4.23,4.49,3.99,2013-14,2013
2,201189,Aaron Gray,1610612758,SAC,36,11,25,9.7,9.7,3586.7,0.7,0.4,0.3,4.23,4.49,3.91,2013-14,2013


---
## Section 5: Pull Team Schedule Data via nba_api

Collect team game schedules to calculate back-to-back games.

**Endpoint:** `TeamGameLog`

**Key fields:** Team ID, game date, home/away, matchup

**Output:** `data/raw/nba_api/team_schedules_{season}.csv` for each season

In [18]:
from nba_api.stats.endpoints import teamgamelog
from nba_api.stats.static import teams

# Get all NBA teams
nba_teams = teams.get_teams()
print(f"Found {len(nba_teams)} NBA teams")

# Display team info
df_teams = pd.DataFrame(nba_teams)
print(df_teams[['id', 'full_name', 'abbreviation']].head(10))

Found 30 NBA teams
           id              full_name abbreviation
0  1610612737          Atlanta Hawks          ATL
1  1610612738         Boston Celtics          BOS
2  1610612739    Cleveland Cavaliers          CLE
3  1610612740   New Orleans Pelicans          NOP
4  1610612741          Chicago Bulls          CHI
5  1610612742       Dallas Mavericks          DAL
6  1610612743         Denver Nuggets          DEN
7  1610612744  Golden State Warriors          GSW
8  1610612745        Houston Rockets          HOU
9  1610612746   Los Angeles Clippers          LAC


In [ ]:
print("Pulling team schedules from nba_api...")
print("="*60)

for season in range(FIRST_SEASON, LAST_SEASON + 1):
    season_str = f"{season}-{str(season+1)[-2:]}"
    filepath = f"../{RAW_NBA_API_DIR}/team_schedules_{season}.csv"
    
    if os.path.exists(filepath):
        print(f"[SKIP] {season_str}: Already exists")
        continue
    
    print(f"[FETCH] {season_str}...")
    
    all_games = []
    for team in nba_teams:
        team_id = team['id']
        team_abbrev = team['abbreviation']
        
        df = safe_api_call(
            teamgamelog.TeamGameLog,
            team_id=team_id,
            season=season_str,
            season_type_all_star='Regular Season'
        )
        
        if not df.empty:
            df['TEAM_ABBREVIATION'] = team_abbrev
            all_games.append(df)
            print(f"  {team_abbrev}: {len(df)} games", end="  ")
            if len(all_games) % 5 == 0:
                print()  # New line every 5 teams
    
    print()  # New line after all teams
    
    if all_games:
        df_season = pd.concat(all_games, ignore_index=True)
        df_season['SEASON'] = season_str
        df_season['SEASON_START_YEAR'] = season
        df_season.to_csv(filepath, index=False)
        print(f"  Saved {len(df_season)} total games for {season_str}")
    else:
        print(f"  No games found for {season_str}")

print("\nDone with team schedules!")

Pulling team schedules from nba_api...
This may take a while due to rate limiting...
[FETCH] 2010-11...
  ATL: 82 games    BOS: 82 games    CLE: 82 games    NOP: 82 games    CHI: 82 games  
  DAL: 82 games    DEN: 82 games    GSW: 82 games    HOU: 82 games    LAC: 82 games  
  LAL: 82 games    MIA: 82 games    MIL: 82 games    MIN: 82 games    BKN: 82 games  
  NYK: 82 games    ORL: 82 games    IND: 82 games    PHI: 82 games    PHX: 82 games  
  POR: 82 games    SAC: 82 games    SAS: 82 games    OKC: 82 games    TOR: 82 games  
  UTA: 82 games    MEM: 82 games    WAS: 82 games    DET: 82 games    CHA: 82 games  

  Saved 2460 total games for 2010-11
[FETCH] 2011-12...
  ATL: 66 games    BOS: 66 games    CLE: 66 games    NOP: 66 games    CHI: 66 games  
  DAL: 66 games    DEN: 66 games    GSW: 66 games    HOU: 66 games    LAC: 66 games  
  LAL: 66 games    MIA: 66 games    MIL: 66 games    MIN: 66 games    BKN: 66 games  
  NYK: 66 games    ORL: 66 games    IND: 66 games    PHI: 66 game

In [20]:
# Verify saved schedule files
print("Saved team schedule files:")
for season in range(FIRST_SEASON, LAST_SEASON + 1):
    filepath = f"../{RAW_NBA_API_DIR}/team_schedules_{season}.csv"
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        print(f"  team_schedules_{season}.csv: {df.shape[0]} games, {df.shape[1]} columns")
    else:
        print(f"  team_schedules_{season}.csv: NOT FOUND")

Saved team schedule files:
  team_schedules_2010.csv: 2460 games, 30 columns
  team_schedules_2011.csv: 1980 games, 30 columns
  team_schedules_2012.csv: 2460 games, 30 columns
  team_schedules_2013.csv: 2460 games, 30 columns
  team_schedules_2014.csv: 2460 games, 30 columns
  team_schedules_2015.csv: 2460 games, 30 columns
  team_schedules_2016.csv: 2460 games, 30 columns
  team_schedules_2017.csv: 2460 games, 30 columns
  team_schedules_2018.csv: 2460 games, 30 columns
  team_schedules_2019.csv: 2118 games, 30 columns


In [21]:
# Preview sample schedule file
sample_file = f"../{RAW_NBA_API_DIR}/team_schedules_{FIRST_SEASON}.csv"
if os.path.exists(sample_file):
    df_sample = pd.read_csv(sample_file)
    print(f"Sample: team_schedules_{FIRST_SEASON}.csv")
    print(f"Columns: {list(df_sample.columns)}")
    print(f"\nFirst 5 rows:")
    display(df_sample.head())

Sample: team_schedules_2010.csv
Columns: ['Team_ID', 'Game_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'TEAM_ABBREVIATION', 'SEASON', 'SEASON_START_YEAR']

First 5 rows:


,Team_ID,Game_ID,GAME_DATE,MATCHUP,WL,W,L,W_PCT,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PTS,TEAM_ABBREVIATION,SEASON,SEASON_START_YEAR
0,1610612737,21001217,"APR 13, 2011",ATL @ CHA,L,44,38,0.537,240,34,79,0.430,1,8,0.125,16,20,0.800,10,27,37,16,3,7,14,15,85,ATL,2010-11,2010
1,1610612737,21001202,"APR 11, 2011",ATL vs. MIA,L,44,37,0.543,240,37,83,0.446,4,12,0.333,12,21,0.571,15,20,35,25,11,1,13,21,90,ATL,2010-11,2010
2,1610612737,21001188,"APR 09, 2011",ATL @ WAS,L,44,36,0.550,240,33,76,0.434,5,9,0.556,12,14,0.857,7,26,33,20,5,7,20,20,83,ATL,2010-11,2010
3,1610612737,21001176,"APR 08, 2011",ATL @ IND,L,44,35,0.557,240,36,76,0.474,5,13,0.385,25,36,0.694,9,27,36,11,6,4,12,21,102,ATL,2010-11,2010
4,1610612737,21001149,"APR 05, 2011",ATL vs. SAS,L,44,34,0.564,240,35,72,0.486,3,13,0.231,17,20,0.850,3,22,25,17,7,2,11,23,90,ATL,2010-11,2010


---
## Summary

### Data Collected

**From elap733 repo (`data/raw/elap733/`):**
| File | Purpose |
|------|---------|
| `missed_games_2010_2019.csv` | **TARGET VARIABLE** - Games missed per injury event |
| `injury_transactions_2010_2019.csv` | IL transaction records (backup) |
| `player_stats_1994_2019.csv` | Historical stats (reference) |
| `team_schedules_2010_2020.csv` | Team schedules for B2B calculation |

**From nba_api (`data/raw/nba_api/`):**
| File | Purpose |
|------|---------|
| `player_stats_{season}.csv` | **MIN, PTS, REB, AST, TOV** - workload features |
| `player_bio_{season}.csv` | **AGE, HEIGHT, WEIGHT** - demographic features |
| `tracking_stats_{season}.csv` | Speed/distance (2013+ only, dropped for coverage) |
| `team_schedules_{season}.csv` | **B2B calculation** - schedule features |

### Data Flow

```
Raw Data (this notebook)
    ↓
02_data_cleaning.ipynb
    → Aggregate games_missed by player-season
    → Create player ID mapping
    → Calculate back-to-back games
    ↓
03_eda.ipynb
    → Feature correlation analysis
    → Identify top predictors
    ↓
04_feature_engineering.ipynb
    → Create age_x_min interaction
    → Create games_missed_last_season lag
    → Create games_missed_next_season target
    ↓
05_supervised_models.ipynb
    → Train/test split (2010-2017 / 2018-2019)
    → Linear Regression, Random Forest, XGBoost
```

### Next Steps
Proceed to `02_data_cleaning.ipynb` to:
1. Clean and standardize player names across datasets
2. Aggregate games missed by player-season for target variable
3. Merge nba_api stats with injury data
4. Calculate back-to-back game features

In [22]:
# Final verification: list all collected files
print("=" * 60)
print("ALL COLLECTED DATA FILES")
print("=" * 60)

print("\nelap733 data:")
for f in sorted(os.listdir(f"../{RAW_ELAP_DIR}")):
    if f.endswith('.csv'):
        size = os.path.getsize(f"../{RAW_ELAP_DIR}/{f}") / 1024
        print(f"  {f}: {size:.1f} KB")

print("\nnba_api data:")
for f in sorted(os.listdir(f"../{RAW_NBA_API_DIR}")):
    if f.endswith('.csv'):
        size = os.path.getsize(f"../{RAW_NBA_API_DIR}/{f}") / 1024
        print(f"  {f}: {size:.1f} KB")

ALL COLLECTED DATA FILES

elap733 data:
  injury_transactions_2010_2019.csv: 873.8 KB
  missed_games_2010_2019.csv: 739.8 KB
  player_stats_1994_2019.csv: 2778.2 KB
  team_schedules_2010_2020.csv: 1918.7 KB

nba_api data:
  player_bio_2010.csv: 62.3 KB
  player_bio_2011.csv: 66.0 KB
  player_bio_2012.csv: 64.8 KB
  player_bio_2013.csv: 66.8 KB
  player_bio_2014.csv: 69.4 KB
  player_bio_2015.csv: 66.1 KB
  player_bio_2016.csv: 67.8 KB
  player_bio_2017.csv: 75.8 KB
  player_bio_2018.csv: 74.3 KB
  player_bio_2019.csv: 74.8 KB
  player_stats_2010.csv: 131.0 KB
  player_stats_2011.csv: 138.7 KB
  player_stats_2012.csv: 136.3 KB
  player_stats_2013.csv: 140.1 KB
  player_stats_2014.csv: 143.3 KB
  player_stats_2015.csv: 138.7 KB
  player_stats_2016.csv: 142.0 KB
  player_stats_2017.csv: 157.8 KB
  player_stats_2018.csv: 155.1 KB
  player_stats_2019.csv: 154.9 KB
  team_schedules_2010.csv: 340.3 KB
  team_schedules_2011.csv: 273.5 KB
  team_schedules_2012.csv: 340.6 KB
  team_schedules_201